### Multidimensional GP

Until now, our examples have been of 1-dimensional Gaussian processes, where there is just a single predictor variable thought to have a non-linear relationship to the outcome. Let's look at a real-world dataset that involves two predictors. We will use the famous **Walker Lake dataset (Isaaks & Srivistava 1989)** that involves spatial sampling of minerals and other variables over space. The data consist of two spatial coordinates and three measured outcomes. The outcomes are anonymously labeled as U, V (continuous variables, such as concentrarion) and T (discrete variable, such as the presence of a particular element).

In [1]:
import os

# os.environ['MKL_NUM_THREADS'] = '1'
# os.environ['OPENBLAS_NUM_THREADS'] = '1'
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.simplefilter('ignore')
import ipywidgets as widgets
from IPython.display import display
import pickle as pkl

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
colab = True
if colab:
  walker_data = pd.read_table("/content/drive/My Drive/Bayesian_programming/data/walker.txt", sep='\s+', index_col=0, skiprows=8, header=None,
              names=['ID', 'Xloc', 'Yloc', 'V', 'U', 'T'])
else:
  walker_data = pd.read_table('../data/walker.txt', sep='\s+', index_col=0, skiprows=8, header=None,
              names=['ID', 'Xloc', 'Yloc', 'V', 'U', 'T'])


X_obs = walker_data[['Xloc', 'Yloc']].values
y_obs = walker_data.V.values

The samples are taken regularly over a coarse grid across the entire area, and then irregularly over portions of the area, presumably where there were positive samples on the coarser grid.

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_obs, y_obs, test_size=0.2, random_state=42)


In [5]:
import plotly.graph_objects as go
import numpy as np

# Generate some sample data

# Create a 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=X_train[:, 0], y=X_train[:, 1], z=y_train,
    mode='markers',
    marker=dict(
        size=5,
        color=y_train,                # Set color to the z values
        colorscale='Viridis',   # Choose a colorscale
        opacity=0.8
    )
)])

# Set labels for axes
fig.update_layout(
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Minerals'
    ),
    title='Interactive 3D Scatter Plot'
)

# Show the plot
fig.show()


We need a sparse grid of inducing points:

In [ ]:
with pm.Model() as spatial_model:


    ## Defining the priors
    l = pm.HalfCauchy("l", beta=3, shape=(2,))
    sf2 = pm.HalfCauchy("sf2", beta=3)

    ## 6. Defining the kernel
    kenral_option = 2
    # option 1: expquad
    if kenral_option == 1:
      K = pm.gp.cov.ExpQuad(2, l) * sf2**2
    elif kenral_option == 2:
      # option 2: Matern32
      K = pm.gp.cov.Matern32(2, l) * sf2**2
    elif kenral_option == 3:
      # option 3: Matern 52
      K = pm.gp.cov.Matern52(2, l) * sf2**2


    ## Defning mean vector
    M = pm.gp.mean.Constant(c=y_obs.mean())

    ## 4. Defining the GP process
    gp = pm.gp.Marginal(mean_func=M, cov_func=K)

    ## 3. Prior of the noise
    sn2 = pm.HalfCauchy("sn2", beta=3)

    ## 2. Defining the likelihood
    obs = gp.marginal_likelihood("obs", X=X_train, y=y_train, noise=sn2)

    ## 1. Instead of sampling mcmc we use find_MAP - give us an optimal value from the unknown posterior
    mp = pm.find_MAP()

Output()

## Lets check MSE under the test set

In [11]:
with spatial_model:

    # The mean and full covariance
    mu_preds, cov_preds = gp.predict(X_test, point=mp)

((mu_preds - y_test)**2).mean()

np.float64(51333.45703877365)

## It looks like the Matern3/2 has the best MSE. This should not surprize us as the data is very jagged.

In [ ]:
mp

## Lets plot the GP plain against the test set points

In [12]:
nd = 30
z1, z2 = np.meshgrid(np.linspace(0, 300, nd), np.linspace(0, 300, nd))
Z = np.concatenate([z1.reshape(nd*nd, 1), z2.reshape(nd*nd, 1)], 1)
Z.shape

(900, 2)

In [ ]:
Z

## We need to make preditions over a grid. We run predicitons for values as defined above under Z.

## Let's make predictions according to mean values under Z.

## For this purpose we use the .predict function as can be seen below.



In [13]:
with spatial_model:
    # The mean and full covariance
    mu, cov = gp.predict(Z, point=mp)

In [14]:
# Create a 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=X_test[:, 0], y=X_test[:, 1], z=y_test,
    mode='markers',
    marker=dict(
        size=15,
        color=y_test,                # Set color to the z values
        colorscale='Viridis',   # Choose a colorscale
        opacity=0.8
    )
)])


# Create a 3D scatter plot
scatter = go.Scatter3d(
    x=X_test[:, 0], y=X_test[:, 1], z=y_test,
    mode='markers',
    marker=dict(
        size=5,
        color=y_test,             # Set color to the z values
        colorscale='Viridis',        # Choose a colorscale
        opacity=0.8
    ),
    name='Scatter Points'
)

surface = go.Surface(
    x=z1, y=z2, z=mu.reshape(30,30),
    colorscale='Viridis',
    opacity=0.7,
    name='Surface'
)
fig = go.Figure(data=[scatter, surface])
# Set labels for axes
fig.update_layout(
    scene=dict(
        xaxis_title='X ',
        yaxis_title='Y ',
        zaxis_title='Mineral'
    ),
    title='GP regressior test set',
    width=1400,  # Adjust width here
    height=800
)

# Show the plot
fig.show()

## What is the probability that the mineral in corrdinate in X = 150 and Y = 200 will be larger than 200?

In [15]:
with spatial_model:

    # The mean and full covariance
    mu_preds, cov_preds = gp.predict(np.array([[150,200]]), point=mp)

mu_preds, cov_preds

(array([185.03134794]), array([[37038.51528012]]))

In [16]:
from scipy.stats import norm
score = 200
mean = mu_preds[0]
std_dev = np.sqrt(cov_preds[0])

z_score = (score - mean) / std_dev

# Calculate the probability of getting a score less than 60
prob = 1-norm.cdf(z_score)
prob

array([0.46900238])

For comperison let use linear regression and see which method has a smaller MSE (hint: it is going to be GP :))

In [17]:
data_lin_reg = pd.DataFrame(columns = ['X', 'Y'])
data_lin_reg['X'] = X_train[:,0]
data_lin_reg['Y'] = X_train[:,1]
data_lin_reg['intercept'] = np.ones(data_lin_reg.shape[0])
data_lin_reg['mineral'] = y_train
data_lin_reg.head()

,X,Y,intercept,mineral
0,44,180,1.0,876.4
1,161,241,1.0,141.9
2,28,150,1.0,335.0
3,160,218,1.0,580.4
4,71,29,1.0,673.3


In [18]:
import statsmodels.api as sm
# Create a linear regression model
model = sm.OLS(data_lin_reg['mineral'], data_lin_reg[['intercept', 'X', 'Y']])
# Fit the model
results = model.fit()
# Print summary statistics, including p-values for each coefficient
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                mineral   R-squared:                       0.075
Model:                            OLS   Adj. R-squared:                  0.070
Method:                 Least Squares   F-statistic:                     15.16
Date:                Tue, 13 May 2025   Prob (F-statistic):           4.70e-07
Time:                        12:54:07   Log-Likelihood:                -2654.1
No. Observations:                 376   AIC:                             5314.
Df Residuals:                     373   BIC:                             5326.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
intercept    596.9273     37.477     15.928      0.0

In [19]:
results.params['intercept'], results.params['X'], results.params['Y']

(np.float64(596.9273127189625),
 np.float64(-1.0396094772776392),
 np.float64(-0.31461179577763865))

In [20]:
def linear_surface_eq(X1, X2):
    return results.params['intercept']+results.params['X']*X1 + results.params['Y']*X2


In [21]:
data_lin_reg_test = pd.DataFrame(columns = ['X', 'Y'])
data_lin_reg_test['X'] = X_test[:,0]
data_lin_reg_test['Y'] = X_test[:,1]
data_lin_reg_test['mineral'] = y_test
data_lin_reg_test.head()

,X,Y,mineral
0,69,208,97.4
1,88,269,12.1
2,49,68,37.8
3,93,41,395.5
4,36,219,429.1


In [22]:
Z = linear_surface_eq(z1, z2)
Z.shape, z1.shape, z2.shape

# Create a 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=X_test[:, 0], y=X_test[:, 1], z=y_test,
    mode='markers',
    marker=dict(
        size=15,
        color=y_test,                # Set color to the z values
        colorscale='Viridis',   # Choose a colorscale
        opacity=0.8
    )
)])


# Create a 3D scatter plot
scatter = go.Scatter3d(
    x=X_test[:, 0], y=X_test[:, 1], z=y_test,
    mode='markers',
    marker=dict(
        size=5,
        color=y_test,             # Set color to the z values
        colorscale='Viridis',        # Choose a colorscale
        opacity=0.8
    ),
    name='Scatter Points'
)

surface = go.Surface(
    x=z1, y=z2, z=Z,
    colorscale='Viridis',
    opacity=0.7,
    name='Surface'
)
fig = go.Figure(data=[scatter, surface])
# Set labels for axes
fig.update_layout(
    scene=dict(
        xaxis_title='X ',
        yaxis_title='Y ',
        zaxis_title='Mineral'
    ),
    title='Linear regressior test set',
    width=1400,  # Adjust width here
    height=800
)

# Show the plot
fig.show()

## Computing MSE for the linear regression

In [23]:
lin_preds  = linear_surface_eq(X_test[:,0], X_test[:,1])

In [25]:
((lin_preds - y_test)**2).mean().item()

104981.9393439271

## A two dimentional problem

### You are managing a software company and in charged of the server capacity.
### You have different regions (where each region differs by the size of activity) in which you need to supply servers for 24 hours a day. The demand for servers varies with time and region size.

### You have the ability to purchase servers from AWS, however, you need to do it before the demand reveals.

### The decision is need to be made arbitrary over time and for different region.

### For example, the hour is now 05:00 AM, and you need to assign servers for a region which has 20K customers. What is the server capacity the you need to purchase?

### Every capacity unit cost 10 Dollars. However, the damage for not being able to supply a single demand cost 30.

### You have previuois data, which reveals past demand at differt hour of the day and region size.

### Based on the data, what should be the optimal assignment strategy?

### We first try to estimate the demand the best way we can.

### Then, we take a simple approach, we assign supply according to the mean value of our prediction, which would work decently.

### We'll also try a linear regression and compare both of them.

## Procedure:

1. Estimating the demand function.
2. Computing the cost function

## Ground Truth

The true demand function looks like this

In [26]:
data, X_data = pkl.load(open('/content/drive/My Drive/Bayesian_programming/data/data_demand.pkl', 'rb'))

In [27]:
x = np.arange(0, 24, 0.2)
y = np.arange(1, 50, 0.1)

X, Y = np.meshgrid(x,y)
Z_demand = X*np.sin(X/1.5)*Y**0.2+ 120 + np.random.normal(0,1.5)



# Create a 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=X_data[:,0], y=X_data[:,1], z=data,
    mode='markers',
    marker=dict(
        size=15,
        color=data,                # Set color to the z values
        colorscale='Viridis',   # Choose a colorscale
        opacity=0.8
    )
)])


# Create a 3D scatter plot
scatter_demand = go.Scatter3d(
    x=X_data[:,0], y=X_data[:,1], z=data,
    mode='markers',
    marker=dict(
        size=5,
        color=y_test,             # Set color to the z values
        colorscale='Viridis',        # Choose a colorscale
        opacity=0.8
    ),
    name='Scatter Points'
)

surface_demand = go.Surface(
    x=X, y=Y, z=Z_demand,
    colorscale='Viridis',
    opacity=0.7,
    name='Surface'
)
fig = go.Figure(data=[scatter_demand, surface_demand])
# Set labels for axes

fig.update_layout(
    scene=dict(
        xaxis_title='Hour',
        yaxis_title='Size',
        zaxis_title='Demand', aspectmode='manual',  # Manually set the aspect ratio
        aspectratio=dict(x=1, y=1, z=1)
    ),
    title='Demand',
    width=800,  # Adjust width here
    height=800
)

# Show the plot
fig.show()

## Defining a pymc GP model:
### Lets do it buttom to up

### 1. Sampling the posterior
### 2. Defining the likelihood: using gp.marginal_likelihood
### 3. Defining prior to the noise parameter
### 4. Defining the Pymc gp object: this includes both mean function and covariance function
### 5. defining the mean function
### 6. defining the covariance function
### 7. defining prior of the cov function

## Fitting a GP

In [28]:
with pm.Model() as spatial_model_1:

    l = pm.HalfCauchy("l", beta=3, shape=(2,))
    sf2 = pm.HalfCauchy("sf2", beta=3)

    # 6 defining the covariance function
    K = pm.gp.cov.ExpQuad(2, l) * sf2**2

    # 5 defining the mean function
    M = pm.gp.mean.Constant(data.mean())

    # 4 Defining the Pymc gp object: this includes both mean function and covariance function
    gp_spatial = pm.gp.Marginal(mean_func=M, cov_func=K)

    # 3 Defining prior to the noise parameter
    sn2 = pm.HalfCauchy("sn2", beta=3)


    # 2 Defining the likelihood: using gp.marginal_likelihood
    obs = gp_spatial.marginal_likelihood("obs", X=X_data,  y=data, noise = sn2)

    # 1 Sampling the posterior
    mp_res_alloc = pm.find_MAP()


Output()

In [ ]:
mp_res_alloc

{'l_log__': array([1.02556019, 4.13693347]),
 'sf2_log__': array(3.165103),
 'sn2_log__': array(0.3470776),
 'l': array([ 2.78865721, 62.61052987]),
 'sf2': array(23.69118398),
 'sn2': array(1.41492652)}

### Sampling a single realization from the posterior

In [29]:
nd = 30
z1a, z2a = np.meshgrid(np.linspace(0, 24, nd), np.linspace(1, 50, nd))
Z1 = np.concatenate([z1a.reshape(nd*nd, 1), z2a.reshape(nd*nd, 1)], 1)
Z1.shape

(900, 2)

## Lets find the GP prediciton using the .predict function:

In [30]:

with spatial_model_1:
  mu1, cov1 = gp_spatial.predict(Z1, point=mp_res_alloc)


## Visualizing the results

In [31]:
x = np.arange(0, 24, 0.2)
y = np.arange(1, 50, 0.1)

X, Y = np.meshgrid(x,y)
Z_demand = X*np.sin(X/1.5)*Y**0.2+ 120 + np.random.normal(0,1.5)

# Create a 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=X_data[:,0], y=X_data[:,1], z=data,
    mode='markers',
    marker=dict(
        size=15,
        color=data,                # Set color to the z values
        colorscale='Viridis',   # Choose a colorscale
        opacity=0.8
    )
)])


# Create a 3D scatter plot
scatter_demand = go.Scatter3d(
    x=X_data[:,0], y=X_data[:,1], z=data,
    mode='markers',
    marker=dict(
        size=5,
        color=y_test,             # Set color to the z values
        colorscale='Viridis',        # Choose a colorscale
        opacity=0.8
    ),
    name='Scatter Points'
)

surface_demand = go.Surface(
    x=X, y=Y, z=Z_demand,
    colorscale='Viridis',
    opacity=0.7,
    name='Surface Ground truth'
)

surface_demand_gp = go.Surface(
    x=z1a, y=z2a, z=mu1.reshape(30,30),
    colorscale='plasma',
    opacity=0.7,
    name='Surface GP'
)


fig = go.Figure(data=[scatter_demand, surface_demand, surface_demand_gp])
# Set labels for axes

fig.update_layout(
    scene=dict(
        xaxis_title='Hour',
        yaxis_title='Size',
        zaxis_title='Demand', aspectmode='manual',  # Manually set the aspect ratio
        aspectratio=dict(x=1, y=1, z=1)
    ),
    title='Demand',
    width=700,  # Adjust width here
    height=700,
    showlegend=True
)

# Show the plot
fig.show()

## Linear regression

Lets try do it also with Linear regression

In [ ]:
X_data[:,0]

In [32]:
with pm.Model() as linear_assignment:

    β = pm.Normal('β', mu=0, sigma=50, shape=3)
    σ = pm.HalfNormal('σ', sigma=50)

    μ = β[0] + β[1] * X_data[:,0] + β[2] * X_data[:,1]

    recruits = pm.Normal('recruits', mu=μ, sigma=σ, observed=data)

    quad_trace = pm.sample(1000, tune=500, chains = 2, cores=2, random_seed=1)

Output()

### Aranging the predicition for visualization in a surface:

In [33]:
samples_linear = quad_trace.posterior['β']
betas = np.array(samples_linear).reshape(-1, 1000, 3)[0,:,:].mean(axis = 0)

lin_reg = []
for x_val in z1a[0,:]:
    for y_val in z2a[:,0]:
        lin_reg.append(betas[0] + betas[1]*x_val + betas[2]*y_val)

lin_reg = np.array(lin_reg).reshape(30,30)

## Visualizing the results

In [34]:
x = np.arange(0, 24, 0.2)
y = np.arange(1, 50, 0.1)

X, Y = np.meshgrid(x,y)
Z_demand = X*np.sin(X/1.5)*Y**0.2+ 120 + np.random.normal(0,1.5)



# Create a 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=X_data[:,0], y=X_data[:,1], z=data,
    mode='markers',
    marker=dict(
        size=15,
        color=data,                # Set color to the z values
        colorscale='Viridis',   # Choose a colorscale
        opacity=0.8
    )
)])


# Create a 3D scatter plot
scatter_demand = go.Scatter3d(
    x=X_data[:,0], y=X_data[:,1], z=data,
    mode='markers',
    marker=dict(
        size=5,
        color=y_test,             # Set color to the z values
        colorscale='Viridis',        # Choose a colorscale
        opacity=0.8
    ),
    name='Scatter Points'
)

surface_demand = go.Surface(
    x=X, y=Y, z=Z_demand,
    colorscale='Viridis',
    opacity=0.7,
    name='Surface'
)

surface_demand_gp = go.Surface(
    x=z1a, y=z2a, z=mu1.reshape(30,30),
    colorscale='plasma',
    opacity=0.7,
    name='Surface'
)
surface_demand_lin = go.Surface(
    x=z1a, y=z2a, z=lin_reg,
    colorscale='plasma',
    opacity=0.7,
    name='Surface'
)


fig = go.Figure(data=[scatter_demand, surface_demand, surface_demand_gp, surface_demand_lin])
# Set labels for axes

fig.update_layout(
    scene=dict(
        xaxis_title='Hour',
        yaxis_title='Size',
        zaxis_title='Demand', aspectmode='manual',  # Manually set the aspect ratio
        aspectratio=dict(x=1, y=1, z=1)
    ),
    title='Demand',
    width=700,  # Adjust width here
    height=700
)

# Show the plot
fig.show()

## Cost function

### Lets follow the assignemt strategy suggested above and compare the total cost based on both linear and gp regressors

In [35]:
def cost_func(demand, supply, hour, size,  a1 = 10, a2 = 30):

    return  a1*supply + a2*max(0, demand-supply)

def lin_demand(betas, hour, size):

   return betas[0] + betas[1]*hour + betas[2]*size


def mean_gp_val(hour, size):
    with spatial_model_1:
        mu1, cov1 = gp_spatial.predict(np.array([[hour,size]]), point=mp_res_alloc)
    return mu1[0]

def demand_true(hour, size):

    return hour * np.sin(hour / 1.5) * size ** 0.2 + 120

In [36]:
gp_tot_cost = []
lin_tot_cost = []

for ind in tqdm(range(50)):

  hour = np.random.uniform(0,24)
  size = np.random.uniform(0,50)

  supplygp = mean_gp_val(hour, size)

  demand = demand_true(hour, size)
  supplylin = lin_demand(betas, hour,size)

  cost_lin = cost_func(demand, supplylin, hour, size)
  cost_gp = cost_func(demand, supplygp, hour, size)

  gp_tot_cost.append(cost_gp)
  lin_tot_cost.append(cost_lin)


print('linear cost: ', np.array(lin_tot_cost).mean() )
print('gp cost: ', np.array(gp_tot_cost).mean())
print('Percetage saved: ', 100*(np.array(lin_tot_cost).mean()-np.array(gp_tot_cost).mean())/np.array(lin_tot_cost).mean())

100%|██████████| 50/50 [00:37<00:00,  1.33it/s]

linear cost:  1450.6233770873037
gp cost:  1248.1248318597643
Percetage saved:  13.959415546861981
